# Tutorial 03 — Sidecar `alexandria-client`: `S1`–`S13` Synchronization & L7 DSP Proxy

In production, a dataspace connector cannot query the Vocabulary Hub inline during every HTTP DSP request: doing so would add network latency, create a single point of failure, and leak traffic metadata.

Instead, **`alexandria-client`** (`C-CLI`) runs as a local sidecar next to the Connector:
1. **Background synchronization (`S1`–`S13`):** Periodically fetches `GET /v1/versions`, verifies the Ed25519 signature over JCS against the Hub's DID Document, enforces anti-rollback continuity, and verifies the `ni:///sha-256;...` digests of every PVP package file.
2. **Hot-path isolation (`REQ-SYS-002`):** Evaluates incoming (`dsp-ingress`) and outgoing (`dsp-egress`) DSP messages strictly in memory against already-verified packages.
3. **Transparent L7 proxy (`REQ-SYS-003`):** Forwards valid messages byte-for-byte to the connector and rejects violations with normative DSP error bodies (`ContractNegotiationError`, `TransferError`, `CatalogError`) and the RFC 9651 `VHub-Verdict` header.

In [1]:
from lab import AlexandriaLab, http_request, show_events_table, show_state_machine

# Configure routes in enforce, shadow, and observe modes to compare their behavior
routes = [
    {
        "listener": "dsp-ingress",
        "match": {"method": "POST", "path": "/negotiations/request"},
        "message": "ContractRequestMessage",
        "gates": ["dsp_message", "policy_intake"],
        "mode": "enforce",
        "body": {"select": "$.offer"},
    },
    {
        "listener": "dsp-ingress",
        "match": {"method": "POST", "path": "/negotiations/offers"},
        "message": "ContractOfferMessage",
        "gates": ["dsp_message", "policy_intake"],
        "mode": "shadow",
        "body": {"select": "$.offer"},
    },
    {
        "listener": "dsp-ingress",
        "match": {"method": "POST", "path": "/catalog/request"},
        "message": "CatalogRequestMessage",
        "gates": ["dsp_message"],
        "mode": "observe",
    },
]

lab = AlexandriaLab(compiler="dsl", initial_clock="2026-09-01T10:00:00Z", routes=routes)
lab.start_connector()
lab.start_server()

# Publish version 1.0.0 on the Hub (with both json-schema and shacl enabled on policy_intake)
draft_v1 = (
    (lab.root / "conformance/f6/drafts/v1.0.0.yaml")
    .read_text(encoding="utf-8")
    .replace(
        "- { role: json-schema, applies_to: policy_intake }",
        "- { role: json-schema, applies_to: policy_intake }\n      - { role: shacl, applies_to: policy_intake }",
    )
)
_, _, d = http_request("POST", f"{lab.hub_url}/v1/drafts", token="editor-token", body=draft_v1, verbose=False)
_, _, c = http_request("POST", f"{lab.hub_url}/v1/drafts/{d['id']}:propose", token="editor-token", verbose=False)
http_request("POST", f"{lab.hub_url}/v1/candidates/{c['id']}:approve", token="releaser-token", body={}, verbose=False)

# Advance the clock to September 5 so 1.0.0 is Enforced (past the P2D propagation window)
lab.set_clock("2026-09-05T12:00:00Z")

# Start alexandria-client and wait for its initial S1-S13 synchronization
lab.start_client()
lab.wait_synced()

✔ alexandria-server listening at http://127.0.0.1:62625 (DID: did:web:127.0.0.1%3A62625)

🕒 Clock updated → 2026-09-05T12:00:00Z

✔ alexandria-client active | dsp-ingress: http://127.0.0.1:62627 | dsp-egress: http://127.0.0.1:62629 | admin: http://127.0.0.1:62630

## 1. Inspecting the Admin Listener (`admin`, `REQ-CLI-019`, `REQ-CLI-020`)

`alexandria-client` exposes its status and observability API **exclusively** on the `admin` listener (never on `dsp-ingress` or `dsp-egress`):
- `GET /status`: operational state (`Active`), last accepted listing, clock skew, and retained packages.
- `GET /versions`: active versions and their lifecycle state (`Enforced`).
- `GET /healthz` and `GET /readyz`.

In [2]:
print("1) Client operational status (GET /status on admin):")
st, _, status_doc = http_request("GET", f"{lab.admin_url}/status")
assert st == 200
assert status_doc["state"] == "Active"
show_state_machine(status_doc["state"], subtitle="Version 1.0.0 in force (head)")

print("2) Synchronized versions (GET /versions on admin):")
st_v, _, _ = http_request("GET", f"{lab.admin_url}/versions")
assert st_v == 200

1) Client operational status (GET /status on admin):
2) Synchronized versions (GET /versions on admin):


connection,close
content-length,464
content-type,application/json
date,"Tue, 10 Nov 2026 06:37:45 GMT"


🛡️ Client operational state: Active (Version 1.0.0 in force (head))

connection,close
content-length,303
content-type,application/json
date,"Tue, 10 Nov 2026 06:37:45 GMT"


## 2. Validating DSP Traffic in `enforce` Mode (`POST /negotiations/request`)

Version `1.0.0` of profile `https://example.com/odrl/profile/f6` permits only the `odrl:use` and `ex:access` actions.
Let's test four HTTP requests through the `dsp-ingress` proxy:
1. **Valid offer with `ex:access`** -> `200 OK`, forwarded to the connector without adding `VHub-Verdict`.
2. **Invalid offer with `odrl:read`** (not present in `1.0.0`) -> `400 Bad Request` blocked at the edge with `VHub-Verdict: violation;gate=policy_intake;code="VERDICT_VIOLATION"` and a `ContractNegotiationError` body.
3. **Offer with an untrusted external `@context`** (`https://evil.example/ctx.jsonld`) -> `400 Bad Request` with `CONTEXT_NOT_IN_PACKAGE` (`REQ-CLI-010`).
4. **Truncated/malformed JSON body** -> `400 Bad Request` with `MESSAGE_UNPARSEABLE` (`REQ-CLI-011`).

In [3]:
def make_contract_request(action: str, extra_ctx: str | None = None) -> dict:
    ctx = [
        "https://example.com/odrl/profile/f6/context.jsonld",
    ]
    if extra_ctx:
        ctx.append(extra_ctx)
    return {
        "@context": ctx,
        "@type": "ContractRequestMessage",
        "consumerPid": "urn:uuid:consumer-001",
        "offer": {
            "@id": "urn:uuid:offer-001",
            "@type": "Offer",
            "profile": "https://example.com/odrl/profile/f6",
            "target": "urn:uuid:dataset-001",
            "permission": [{"action": action}],
        },
        "callbackAddress": "http://consumer.example/callback",
    }

print("Case 1 — Permitted action ex:access (expected 200 OK):")
st1, hdr1, _ = http_request("POST", f"{lab.dsp_ingress_url}/negotiations/request", body=make_contract_request("ex:access"))
assert st1 == 200
assert "vhub-verdict" not in hdr1

print("Case 2 — Disallowed action odrl:read in 1.0.0 (expected 400 VERDICT_VIOLATION):")
st2, hdr2, body2 = http_request("POST", f"{lab.dsp_ingress_url}/negotiations/request", body=make_contract_request("odrl:read"))
assert st2 == 400
assert body2["code"] == "VERDICT_VIOLATION"

print("Case 3 — External @context not declared in the package (expected 400 CONTEXT_NOT_IN_PACKAGE):")
st3, hdr3, body3 = http_request(
    "POST",
    f"{lab.dsp_ingress_url}/negotiations/request",
    body=make_contract_request("ex:access", extra_ctx="https://evil.example/ctx.jsonld"),
)
assert st3 == 400
assert body3["code"] == "CONTEXT_NOT_IN_PACKAGE"

print("Case 4 — Malformed JSON payload (expected 400 MESSAGE_UNPARSEABLE):")
st4, hdr4, body4 = http_request(
    "POST",
    f"{lab.dsp_ingress_url}/negotiations/request",
    headers={"Content-Type": "application/json"},
    body="{invalid-json",
)
assert st4 == 400
assert body4["code"] == "MESSAGE_UNPARSEABLE"

Case 1 — Permitted action ex:access (expected 200 OK):
Case 2 — Disallowed action odrl:read in 1.0.0 (expected 400 VERDICT_VIOLATION):
Case 3 — External @context not declared in the package (expected 400 CONTEXT_NOT_IN_PACKAGE):
Case 4 — Malformed JSON payload (expected 400 MESSAGE_UNPARSEABLE):


connection,close
content-length,55
content-type,application/json
date,"Tue, 10 Nov 2026 06:37:45 GMT"


connection,close
content-length,193
content-type,application/json
date,"Tue, 10 Nov 2026 06:37:45 GMT"
vhub-verdict,"violation;gate=policy_intake;code=""VERDICT_VIOLATION"""


connection,close
content-length,243
content-type,application/json
date,"Tue, 10 Nov 2026 06:37:45 GMT"
vhub-verdict,"violation;gate=policy_intake;code=""CONTEXT_NOT_IN_PACKAGE"""


connection,close
content-length,159
content-type,application/json
date,"Tue, 10 Nov 2026 06:37:45 GMT"
vhub-verdict,"violation;gate=dsp_message;code=""MESSAGE_UNPARSEABLE"", violation;gate=policy_intake;code=""MESSAGE_UNPARSEABLE"""


## 3. Comparing with `shadow` Mode (`REQ-CLI-012`)

On the `POST /negotiations/offers` route we configured `"mode": "shadow"`.
When we send an invalid offer (`odrl:read`) to a `shadow` route:
- The proxy **does not block the traffic** (it forwards it to the connector and returns `200 OK` without a `VHub-Verdict` header).
- However, it **emits the verdict event** in `events.jsonl` so operators can audit what would fail before enabling `enforce`.

In [4]:
shadow_msg = make_contract_request("odrl:read")
shadow_msg["@type"] = "ContractOfferMessage"

print("Request with policy violation on a shadow route (expected 200 OK forwarded to connector):")
st_sh, hdr_sh, _ = http_request("POST", f"{lab.dsp_ingress_url}/negotiations/offers", body=shadow_msg)
assert st_sh == 200
assert "vhub-verdict" not in hdr_sh

Request with policy violation on a shadow route (expected 200 OK forwarded to connector):


connection,close
content-length,54
content-type,application/json
date,"Tue, 10 Nov 2026 06:37:45 GMT"


## 4. Verdict Events (`events.jsonl`) and Prometheus Metrics (`GET /metrics`)

Let's inspect the structured events emitted in `events.jsonl` (conforming to Appendix A.4) and the Prometheus metrics exposed at `GET /metrics`:

In [5]:
events = lab.read_events()
show_events_table(events, title="Events emitted by alexandria-client (events.jsonl)")

print("=== Prometheus Metrics (GET /metrics) ===")
st_m, _, metrics_txt = http_request("GET", f"{lab.admin_url}/metrics")
assert st_m == 200
lab.close()

=== Prometheus Metrics (GET /metrics) ===


Type,Instant (UTC),Mode / Sev,Route / Version,Classification / Agreement,Codes
vhub.state,2026-09-05T12:00:00Z,info,None → Active,-,OK
vhub.sync,2026-09-05T12:00:00Z,error,,-,POC_EVAL_UNSUPPORTED_ROLE
vhub.verdict,2026-09-05T12:00:00Z,enforce,POST /negotiations/request,violation,VERDICT_VIOLATION
vhub.verdict,2026-09-05T12:00:00Z,enforce,POST /negotiations/request,violation,CONTEXT_NOT_IN_PACKAGE
vhub.verdict,2026-09-05T12:00:00Z,enforce,POST /negotiations/request,violation,MESSAGE_UNPARSEABLE
vhub.verdict,2026-09-05T12:00:00Z,enforce,POST /negotiations/request,violation,MESSAGE_UNPARSEABLE
vhub.verdict,2026-09-05T12:00:00Z,shadow,POST /negotiations/offers,violation,VERDICT_VIOLATION


connection,close
content-length,637
content-type,application/openmetrics-text; version=1.0.0; charset=utf-8
date,"Tue, 10 Nov 2026 06:37:45 GMT"
